# Uninstalling dependency clashes and installing gemini

In [24]:
!pip uninstall -qqy jupyterlab kfp
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

In [25]:
from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

'1.7.0'

# IMP: Currently using my API key

In [26]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

In [27]:
client = genai.Client(api_key = GOOGLE_API_KEY)

for m in client.models.list():
    if 'embedContent' in m.supported_actions:
        print(m.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp


## TODO: GET JSON DATA; CONVERT TO DICTIONARY

In [28]:
import json

documents = []
with open('/kaggle/input/blog-posts/blog_posts.jsonl', 'r') as f:
    for line in f:
        documents.append(json.loads(line))

print(type(documents))
print(type(documents[0]))

<class 'list'>
<class 'dict'>


In [29]:
print(documents[0])

{'title': 'In the Kitchen with Nutrition Education', 'url': 'https://www.capitalareafoodbank.org/blog/2025/03/10/in-the-kitchen-with-nutrition-education/', 'date': 'March 10, 2025', 'content': 'Bycafb•March 10, 2025\nThe sounds of chopping and sizzling echo through our training kitchen. Step inside, and you’ll find our Nutrition Education team in action. It’s recipe testing day, a hands-on process that ensures our recipe cards are as delicious and practical as they are nutritious.\nAt the heart of CAFB’s nutrition education efforts are ourrecipe cards,produce guides, and7 Ways cards, which offer new ways to use an ingredient. These resources help make cooking easy and accessible for the communities we serve. Before a recipe is added to our collection, it must pass a few tests. Does it taste good? Are the instructions clear? That’s where recipe testing and development days come in.\nOn this day, the team worked on a variety of dishes, from spicyshakshukato fillingbreakfast burritosand r

In [30]:
text_for_embedding = []
metadata = []

for post in documents:
    text = f"Title: {post['title']} \nContent: {post['content']}"
    text_for_embedding.append(text)
    metadata.append({
        "url": post['url'],
        "date": post['date'],
        "title": post['title']
    })

In [31]:
text_for_embedding[0]

'Title: In the Kitchen with Nutrition Education \nContent: Bycafb•March 10, 2025\nThe sounds of chopping and sizzling echo through our training kitchen. Step inside, and you’ll find our Nutrition Education team in action. It’s recipe testing day, a hands-on process that ensures our recipe cards are as delicious and practical as they are nutritious.\nAt the heart of CAFB’s nutrition education efforts are ourrecipe cards,produce guides, and7 Ways cards, which offer new ways to use an ingredient. These resources help make cooking easy and accessible for the communities we serve. Before a recipe is added to our collection, it must pass a few tests. Does it taste good? Are the instructions clear? That’s where recipe testing and development days come in.\nOn this day, the team worked on a variety of dishes, from spicyshakshukato fillingbreakfast burritosand refreshingbeet salad. Recipes were made following preexisting recipes, or from scratch in a process to develop a new dish. Each dish had

## Embedding

In [32]:
from chromadb import Embeddings, EmbeddingFunction, Documents
from google.api_core import retry

is_retriable = lambda e:(isinstance(e, genai.errors.APIError) and e.code in {429, 503})

class GeminiEmbeddingFunction:
    document_mode = True

    @retry.Retry(predicate = is_retriable)
    def __call__(self, input = Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = 'retrieval_document'
        else:
            embedding_task = 'retrieval_query'

        response = client.models.embed_content(
            model = 'models/text-embedding-004',
            contents = input,
            config = types.EmbedContentConfig(
                task_type = embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

In [33]:
import chromadb

DB_NAME = 'blogdatadb'
embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name = DB_NAME, embedding_function = embed_fn)
for i in range(0, len(text_for_embedding), 98):
    chunk = text_for_embedding[i:i+98]
    ids = [str(j) for j in range(i, i + len(chunk))]
    db.add(documents=chunk, ids=ids)

In [34]:
db.count()

740

In [35]:
db.peek(1)

{'ids': ['0'],
 'embeddings': array([[ 2.59291511e-02, -1.44858025e-02,  3.41253690e-02,
         -9.02335811e-03,  4.44306433e-02, -2.35463884e-02,
          2.47029439e-02,  4.32929676e-03,  2.39903089e-02,
          9.64111276e-03, -2.21332610e-02,  7.08811954e-02,
          6.15286827e-02, -2.53847744e-02, -5.61095253e-02,
         -2.74009909e-02,  6.77281991e-02,  2.87060849e-02,
         -1.07166305e-01, -1.09609161e-02,  3.76998596e-02,
         -5.58131970e-02,  3.68967988e-02, -9.53585133e-02,
         -6.61118999e-02, -7.97188282e-03, -3.35374847e-02,
         -8.38829204e-03,  4.12356015e-03, -5.90689946e-03,
          5.80956228e-02,  1.93206612e-02,  3.23352851e-02,
         -1.60857625e-02,  2.56253313e-03,  6.11905428e-03,
         -3.80914919e-02, -4.89443801e-02,  6.72813058e-02,
         -3.18857357e-02, -7.45240375e-02, -1.64207933e-03,
          1.02803344e-02,  1.65479705e-02, -4.80721705e-02,
         -2.30603777e-02,  3.04219108e-02,  8.48813727e-02,
          4

# Query is also hard coded

# TODO: Take query as input after properly prompt engineering

In [36]:
embed_fn.document_mode = False
query = 'Write a blog post about the rise of food insecurity.'

result = db.query(query_texts = [query], n_results = 3)
[all_passages] = result['documents']
Markdown(all_passages[0])

Title: Hunger Report 2022: 1 in 3 faced food insecurity across region last year 
Content: ByLaura Olson•June 16, 2022
One-third of residents across the Greater Washington region experienced some level of food insecurity in 2021, according to findings froma first-of-its-kind general population survey on food insecurity and inequityconducted by theCapital Area Food BankandNORC at the University of Chicago.

That overall rate means thatmore than 1.2 million people across the region didn’t know where their next meal was coming from at some point during 2021. In some counties, nearly half of all residents experienced food insecurity at some point last year. In every county, at least one in five people faced challenges getting enough to eat.

The breathtaking findings in our new survey are the first publicly available information about the scale of food insecurity in the Greater Washington region during the past year. Together, the responses underscore how the long-standing inequities across our region were exacerbated by the pandemic’s uneven economic effects.
While one-third of local households are earning more than before the public health crisis began, two in three have seen their income remain flat or decline – creating a substantial short-term strain as the cost of food and other necessities have skyrocketed, and the potential for lingering financial effects in the absences of broad action to address inequities.
Read Hunger Report 2022
This year’s analysis is the third Hunger Report conducted by CAFB, following reports in2020and2021. But it is the first in which the food bank has partnered withNORC, one of the largest and most trusted independent social research organizations in the United States. The result is a survey of nearly 4,000 residents across our region, which offers the most detailed look to date at the prevalence and underlying factors of food insecurity locally. (Read more hereon how the survey was conducted.)
Due to differences in methodology, these data cannot be accurately compared to previous studies on regional food security. However, the findings clearly show food insecurity to be much more widespread across the region than previously understood.
Below is a recap of the report’s key findings, as well as recommendations for ways to create greater food security and economic stability for more people in our region.
The sheer prevalence of food insecurity across the regionshown in the survey results is among the most significant takeaways. Usingthe USDA’s standard screener, the survey found33% of individuals across the region were food insecure at some point during the last year. Among that group, half were classified as “severely food insecure,” the most acute tier.
The need for greater access to food affects every community across the region, but not all communities or demographic groups are affected equally. People of color and households with kids are disproportionately affected:

Young adults aged 18–29 also are overrepresentedamong those who are food insecure: they accounted for 31% of the population experiencing food insecurity, despite totaling 21% of the general population.
People experiencing food insecurityare employed at the same rateas those who face challenges accessing enough food: 77% of people facing food insecurity were employed, compared to 74% of those who did not experience food insecurity.
A third of our region is earning less than they were in March 2020.Food-insecure people are significantly overrepresented in this group.

When asked about their own projections of whether their household finances would improve or worsen over the next year, most people across the region are not optimistic.

Most adults agree that governments of all levels bear the largest responsibility for addressing food insecurity in the United States. However, nonprofit organizations are seen as the agencies doing the most to address food insecurity.
The study’s findings indicate there is still a major gap in access to high-quality employment opportunities. More than half of food-insecure adults would experience a major positive impact on their household finances from gaining access to higher-quality jobs, according to the survey findings.
The food bank has a series of pilot initiatives based on this final concept. Known as“Food+” programs, the food bank partners with hospitals and clinics, colleges and universities, and other social service organizations to pair food access with existing programs focused on health care, education and workforce development.
The data are clear: inequity has grown during the pandemic, and the valley between the realities of those at different income levels is set to continue widening. What is needed now are accelerated efforts to address those expanding inequities.

In [37]:
query_oneline = query.replace('\n', ' ')

prompt = f"""
You are a professional blog writer.
Be sure to write in complete sentences, and be comprehensive. Do *NOT* make things up. Do *NOT* hallucinate.
If you feel like you don't have enough information, ASK. 

Below are some documents for your reference.

Do not copy the text directly, instead create an original blog post based on the ideas, themes or writing style you observe.
Maintain a coherent flow, an engaging tone, and a clear structure (Title, Introduction, Body, Conclusion).
If you thing certain parts of the documents are irrelevant, you may ignore it.

QUESTION: {query_oneline}

"""

for passage in all_passages:
    passage_oneline = passage.replace('\n', ' ')
    prompt += f"DOCUMENT: {passage_oneline}\n"

print(prompt)


You are a professional blog writer.
Be sure to write in complete sentences, and be comprehensive. Do *NOT* make things up. Do *NOT* hallucinate.
If you feel like you don't have enough information, ASK. 

Below are some documents for your reference.

Do not copy the text directly, instead create an original blog post based on the ideas, themes or writing style you observe.
Maintain a coherent flow, an engaging tone, and a clear structure (Title, Introduction, Body, Conclusion).
If you thing certain parts of the documents are irrelevant, you may ignore it.

QUESTION: Write a blog post about the rise of food insecurity.

DOCUMENT: Title: Hunger Report 2022: 1 in 3 faced food insecurity across region last year  Content: ByLaura Olson•June 16, 2022 One-third of residents across the Greater Washington region experienced some level of food insecurity in 2021, according to findings froma first-of-its-kind general population survey on food insecurity and inequityconducted by theCapital Area Fo

In [38]:
answer = client.models.generate_content(
    model = 'gemini-2.0-flash',
    contents = prompt
)

Markdown(answer.text)

## The Unsettling Rise of Food Insecurity: A Growing Crisis

Food insecurity, the state of being without reliable access to a sufficient quantity of affordable, nutritious food, is on the rise, casting a shadow over communities and demanding urgent attention. While headlines often tout economic progress, the reality for a significant portion of the population paints a different picture. This blog post delves into the concerning trend of increasing food insecurity, exploring its causes and consequences.

Recent reports reveal a disturbing surge in the number of individuals and families struggling to put food on the table. In the greater Washington region, a recent study showed that a significant percentage of households reported experiencing food insecurity, a notable increase from previous years. This translates to hundreds of thousands of people unsure of where their next meal will come from.

Several factors contribute to this growing crisis. The reports highlight the lingering economic effects of the pandemic, the rollback of expanded government assistance programs, and relentless inflation as major drivers. While some have benefited from economic recovery, many continue to face job losses, reduced hours, and stagnant wages. The rise in the cost of living, especially for necessities like housing and healthcare, further strains already stretched budgets, leaving families with difficult choices. Food often becomes the first area to be cut back, leading to inadequate nutrition and potential health problems.

The consequences of food insecurity extend far beyond hunger. Children experiencing food insecurity may face developmental delays and health issues, impacting their future potential. Adults may struggle with chronic diseases and mental health challenges, affecting their productivity and overall well-being. The economic impact is also significant, as food insecurity can lead to increased healthcare costs and decreased workforce participation.

Addressing this crisis requires a multi-pronged approach involving government, non-profit organizations, and the private sector. Strengthening social safety nets, increasing access to affordable housing and healthcare, and promoting job training and employment opportunities are crucial steps. Additionally, supporting food banks and other charitable organizations that provide immediate relief to those in need is essential.

The rise of food insecurity is a wake-up call. It demands a collective effort to ensure that everyone has access to nutritious food and the opportunity to thrive. By understanding the causes and consequences of this crisis and working together to implement effective solutions, we can build a more food-secure and equitable future for all.
